<a href="https://colab.research.google.com/github/laurakeita/MMM/blob/main/notebooks/06_attribution_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 · Attribution Validation
Compare Meridian's estimated ROI against the known ground-truth ROI used to generate the synthetic dataset.

*Requires `mmm` and `media_summary` from `05_synthetic_validation` + `03_model_diagnostics`.*

In [ ]:
# ==========================================
# Attribution Validation
# ==========================================

print("\n" + "="*50)
print("ATTRIBUTION VALIDATION")
print("="*50)

# Ground Truth ROI from synthetic dataset
true_roi = {
    "Meta": 2.07,
    "Google": 3.00
}

# Meridian estimated ROI
summary_df = media_summary.summary_table()

print("\nEstimated ROI:")
print(summary_df)

roi_validation_results = []

for channel in true_roi.keys():

    try:
        # Filter for posterior distribution for the current channel
        channel_posterior_summary = summary_df[
            (summary_df["distribution"] == "posterior") &
            (summary_df["channel"].str.contains(channel, case=False, na=False))
        ]

        if not channel_posterior_summary.empty:
            roi_str = channel_posterior_summary["roi"].iloc[0]
            estimated_roi = float(roi_str.split(' ')[0])

            error_pct = (
                abs(estimated_roi - true_roi[channel])
                / true_roi[channel]
            ) * 100

            roi_validation_results.append({
                "channel": channel,
                "true_roi": true_roi[channel],
                "estimated_roi": estimated_roi,
                "recovery_error_pct": round(error_pct,2)
            })
        else:
            print(f"Could not find posterior ROI for {channel}")

    except Exception as e:
        print(f"An error occurred while processing ROI for {channel}: {e}")

validation_df = pd.DataFrame(
    roi_validation_results
)

print("\nROI Recovery Validation")
print(validation_df)

avg_error = validation_df[
    "recovery_error_pct"
].mean()

print(
    f"\nAverage ROI Recovery Error: {avg_error:.2f}%"
)